# P1 – Onderzoek van de Database
## Onderdeel 2: Visuele Analyse van Cijfers

**Doel:** Begrijpen waarom classificatie lastig is door het visueel verkennen van de MNIST-dataset.

**Vragen:**
- Laat 5 willekeurige voorbeelden zien van éénzelfde cijfer (bijv. 3)
- Wat valt op?
- Welke cijfers lijken visueel sterk op elkaar?
- Zijn sommige cijfers slordiger geschreven dan andere?
- Vergelijk bijv. 4 en 9, of 1 en 7

## 1. Imports & Dataset laden

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

# MNIST laden
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Trainingsset: {X_train.shape} afbeeldingen, labels: {y_train.shape}")
print(f"Testset:      {X_test.shape} afbeeldingen, labels: {y_test.shape}")
print(f"Pixelwaarden: min={X_train.min()}, max={X_train.max()}")
print(f"Klassen (cijfers): {np.unique(y_train)}")

## 2. Eén voorbeeld per cijfer (0–9)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Eén voorbeeld per cijfer (0–9)", fontsize=14, fontweight="bold")

for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    ax = axes[digit // 5][digit % 5]
    ax.imshow(X_train[idx], cmap="gray")
    ax.set_title(f"Cijfer: {digit}", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Vijf willekeurige voorbeelden van het cijfer 3

In [ ]:
def toon_vijf_voorbeelden(digit, dataset_X, dataset_y, seed=42):
    np.random.seed(seed)
    indices = np.where(dataset_y == digit)[0]
    gekozen = np.random.choice(indices, 5, replace=False)

    fig, axes = plt.subplots(1, 5, figsize=(12, 3))
    fig.suptitle(f"5 willekeurige voorbeelden van cijfer '{digit}'", fontsize=13, fontweight="bold")

    for i, idx in enumerate(gekozen):
        axes[i].imshow(dataset_X[idx], cmap="gray")
        axes[i].set_title(f"Index: {idx}", fontsize=9)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

toon_vijf_voorbeelden(3, X_train, y_train)

### Observaties – Cijfer 3

- De **breedte en hoogte** van de 3 varieert sterk: sommige zijn smal en lang, andere breed en kort.
- De **bovenste boog** kan kleiner of groter zijn dan de onderste boog.
- Sommige 3'en hebben een duidelijk **middenpunt** (de inkeping links), andere bijna niet.
- De **dikte van de lijnen** verschilt: dun vs. dik handschrift.
- De **kanteling** varieert: sommige staan recht, andere hellen naar links of rechts.

## 4. Vergelijking: 4 vs. 9 en 1 vs. 7

In [ ]:
def vergelijk_paar(digit_a, digit_b, dataset_X, dataset_y, n=5, seed=42):
    np.random.seed(seed)
    idx_a = np.random.choice(np.where(dataset_y == digit_a)[0], n, replace=False)
    idx_b = np.random.choice(np.where(dataset_y == digit_b)[0], n, replace=False)

    fig, axes = plt.subplots(2, n, figsize=(14, 5))
    fig.suptitle(f"Vergelijking: {digit_a} (boven) vs. {digit_b} (onder)", fontsize=13, fontweight="bold")

    for i in range(n):
        axes[0][i].imshow(dataset_X[idx_a[i]], cmap="gray")
        axes[0][i].set_title(str(digit_a), fontsize=14, color="steelblue")
        axes[0][i].axis("off")
        axes[1][i].imshow(dataset_X[idx_b[i]], cmap="gray")
        axes[1][i].set_title(str(digit_b), fontsize=14, color="tomato")
        axes[1][i].axis("off")

    plt.tight_layout()
    plt.show()

vergelijk_paar(4, 9, X_train, y_train)
vergelijk_paar(1, 7, X_train, y_train)

### Observaties – 4 vs. 9

- Beide hebben een **gesloten bovenlus** – dit is de kern van de verwarring.
- Bij een slordige 4 (waarbij de bovenkant dicht zit) lijkt het sterk op een 9.
- Het verschil zit in de **verticale lijn**: bij een 4 gaat deze door de lus; bij een 9 hangt de staart eronder.

### Observaties – 1 vs. 7

- Sommige **1'en hebben een schuine bovenkap** – dan lijkt het bijna op een 7.
- Het verschil zit in de **horizontale bovenlijn** van de 7, die bij een 1 ontbreekt.
- Bij de kleine 28×28 resolutie verdwijnen subtiele details snel.

## 5. Variatie in schrijfstijl per cijfer

In [ ]:
std_per_cijfer = []

for digit in range(10):
    subset = X_train[y_train == digit].astype(float)
    gemiddelde_std = subset.std(axis=(1, 2)).mean()
    std_per_cijfer.append(gemiddelde_std)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(range(10), std_per_cijfer, color="steelblue", edgecolor="navy", alpha=0.8)
ax.set_xlabel("Cijfer", fontsize=12)
ax.set_ylabel("Gem. pixelstandaardafwijking", fontsize=12)
ax.set_title("Variatie in schrijfstijl per cijfer\n(hogere waarde = meer variatie)", fontsize=13)
ax.set_xticks(range(10))

for bar, val in zip(bars, std_per_cijfer):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{val:.1f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

meest = np.argmax(std_per_cijfer)
minst = np.argmin(std_per_cijfer)
print(f"Meest variabel: cijfer {meest} (std={std_per_cijfer[meest]:.2f})")
print(f"Minst variabel: cijfer {minst} (std={std_per_cijfer[minst]:.2f})")

## 6. Gemiddeld prototype per cijfer

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Gemiddeld prototype per cijfer", fontsize=13, fontweight="bold")

for digit in range(10):
    subset = X_train[y_train == digit].astype(float)
    gemiddeld = subset.mean(axis=0)
    ax = axes[digit // 5][digit % 5]
    ax.imshow(gemiddeld, cmap="gray")
    ax.set_title(f"Cijfer: {digit}\n(n={len(subset):,})", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Conclusies

**Waarom is classificatie lastig?**

1. **Grote intra-klasse variatie** – Eén cijfer wordt op totaal verschillende manieren geschreven.
2. **Visueel gelijkende paren** – 4 vs. 9, 1 vs. 7, 3 vs. 8, 5 vs. 6.
3. **Lage resolutie (28×28)** – Subtiele details gaan verloren of worden ruis.
4. **Sommige cijfers zijn inconsistenter** – Dit zijn de lastigere klassen voor een classifier.